In [1]:
# Imports & Configuration
from pathlib import Path

import duckdb
import polars as pl
from great_tables import GT

# Change only this path
SOURCE = Path("../../raw_data/raw_budget.csv").resolve().as_posix()

# DuckDB connection
con = duckdb.connect()

In [2]:
# ============================================================
# Functions
# ============================================================

def show_sql(query: str, max_rows: int = 100):
    """Execute DuckDB SQL and display results using Great Tables."""
    result = con.sql(query).limit(max_rows)

    df = pl.DataFrame(
        result.fetchall(),
        schema=result.columns,
        orient="row"
    )

    display(GT(df))


In [3]:
# 01 | Source Overview

show_sql(f"""
    SUMMARIZE
    SELECT *
    FROM read_csv('{SOURCE}', all_varchar = true)
""")

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
month,VARCHAR,2026-01,2026-09,9,None,None,None,None,None,27,0.00
shop,VARCHAR,CZ,SK,3,None,None,None,None,None,27,0.00
orders_budget,VARCHAR,1005,976,25,None,None,None,None,None,27,0.00
net_sales_czk_budget,VARCHAR,1591000,944000,26,None,None,None,None,None,27,0.00


In [4]:
# 02 | Data Quality Checks

show_sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(DISTINCT (month, shop)) AS duplicate_rows,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(month), '') IS NULL
               OR NULLIF(TRIM(shop), '') IS NULL
               OR NULLIF(TRIM(orders_budget), '') IS NULL
               OR NULLIF(TRIM(net_sales_czk_budget), '') IS NULL
        ) AS missing_required,

        COUNT(*) FILTER (
            WHERE TRY_CAST(month || '-01' AS DATE) IS NULL
        ) AS invalid_month,

        COUNT(*) FILTER (
            WHERE shop NOT IN ('CZ', 'SK', 'HU')
        ) AS invalid_shop,

        COUNT(*) FILTER (
            WHERE TRY_CAST(orders_budget AS INTEGER) IS NULL
               OR TRY_CAST(orders_budget AS INTEGER) < 0
               OR TRY_CAST(net_sales_czk_budget AS DECIMAL(18,2)) IS NULL
               OR TRY_CAST(net_sales_czk_budget AS DECIMAL(18,2)) < 0
        ) AS invalid_budget

    FROM read_csv('{SOURCE}', all_varchar = true)
""")

total_rows,duplicate_rows,missing_required,invalid_month,invalid_shop,invalid_budget
27,0,0,0,0,0


In [5]:
# 03 | Budget Coverage

show_sql(f"""
    SELECT
        shop,
        COUNT(*) AS months,
        MIN(month) AS first_month,
        MAX(month) AS last_month,
        SUM(CAST(orders_budget AS BIGINT)) AS budget_orders,
        SUM(CAST(net_sales_czk_budget AS DECIMAL(18,2))) AS budget_sales_czk
    FROM read_csv('{SOURCE}', all_varchar = true)
    GROUP BY shop
    ORDER BY shop
""")

shop,months,first_month,last_month,budget_orders,budget_sales_czk
CZ,9,2026-01,2026-09,8614,17377000.00
HU,9,2026-01,2026-09,1775,3149000.00
SK,9,2026-01,2026-09,3401,6732000.00


| Area | Finding | Recommendation |
|---|---|---|
| Uniqueness | 27 records, no duplicate month × shop combinations | Preserve source grain |
| Completeness | All 3 shops have budgets for 9 months (Jan–Sep 2026) | No missing periods detected |
| Data quality | No missing or invalid values | Apply appropriate data types |
| Budget values | Valid order counts and net sales amounts | Preserve original budget values |
| Currency | Net sales budgets expressed in CZK | No FX conversion required |

**Grain:** One row per month × shop.

**Staging:** Standardize month, shop and numeric data types.

**Business logic:** Compare actual sales and order counts against monthly budgets.